# Hospital Flow Burden — 00 - Executive Summary

**Author:** Saige Mukherjee  
**Contact:** mukherjeesaige@gmail.com  
**LinkedIn:** https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside one collapsed generator cell. The reader needs to read the continuous report produced beneath it: project purpose, headline findings, KPI cards, graph explanations, and operational implications.

To produce the report:

1. Obtain credentialed access to MIMIC-IV through PhysioNet and BigQuery.
2. Create a Google Cloud project authorized to query the PhysioNet-hosted dataset.
3. Enter the billing project ID below and execute the program.

In [ ]:
BILLING_PROJECT = ""

In [ ]:
import html
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, clear_output, display
from google.cloud import bigquery


HOSP_DATASET = "physionet-data.mimiciv_3_1_hosp"
MAX_SEGMENT_HOURS = 24 * 365
EXCESS_THRESHOLD_HOURS = 72
TOP_N_CAREUNITS = 8
MIN_CELL_N = 11
MAX_BYTES_BILLED = 5_000_000_000

client = bigquery.Client(project=BILLING_PROJECT)
job_config = bigquery.QueryJobConfig(maximum_bytes_billed=MAX_BYTES_BILLED)

pd.set_option("display.max_columns", 50)
plt.rcParams.update({
    "figure.figsize": (9, 4.8),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})


def query_df(sql: str) -> pd.DataFrame:
    """Run an aggregate BigQuery query and return a pandas DataFrame."""
    return client.query(sql, job_config=job_config).to_dataframe(
        create_bqstorage_client=False
    )

def format_n(n):
    """Helper to mask counts less than 11."""
    try:
        val = int(n)
        return "<11" if val < MIN_CELL_N else f"{val:,}"
    except (ValueError, TypeError):
        return str(n)

BASE_SEGMENTS_CTE = f"""
raw_segments AS (
  SELECT
    hadm_id,
    transfer_id,
    careunit,
    intime,
    outtime,
    DATETIME_DIFF(outtime, intime, SECOND) / 3600.0 AS duration_hours
  FROM `{HOSP_DATASET}.transfers`
  WHERE careunit IS NOT NULL
    AND intime IS NOT NULL
    AND outtime IS NOT NULL
),
segments AS (
  SELECT *
  FROM raw_segments
  WHERE duration_hours > 0
    AND duration_hours <= {MAX_SEGMENT_HOURS}
),
admission_segments AS (
  SELECT *
  FROM segments
  WHERE hadm_id IS NOT NULL
)
"""

kpi_sql = f"""
WITH
{BASE_SEGMENTS_CTE}
SELECT
  COUNT(*) AS segment_n,
  COUNTIF(hadm_id IS NOT NULL) AS admission_linked_segment_n,
  COUNTIF(hadm_id IS NULL) AS non_admission_segment_n,
  COUNT(DISTINCT hadm_id) AS admission_n,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(50)] AS median_hours,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(75)] AS p75_hours,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(90)] AS p90_hours,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(95)] AS p95_hours,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(99)] AS p99_hours,
  100 * SAFE_DIVIDE(COUNTIF(duration_hours > 48), COUNT(*)) AS pct_gt_48h,
  100 * SAFE_DIVIDE(COUNTIF(duration_hours > 72), COUNT(*)) AS pct_gt_72h,
  100 * SAFE_DIVIDE(COUNTIF(duration_hours > 168), COUNT(*)) AS pct_gt_7d,
  100 * SAFE_DIVIDE(COUNTIF(duration_hours > 336), COUNT(*)) AS pct_gt_14d,
  SUM(duration_hours) AS total_segment_hours,
  SUM(GREATEST(duration_hours - {EXCESS_THRESHOLD_HOURS}, 0)) AS excess_72h,
  (SELECT COUNTIF(duration_hours > {MAX_SEGMENT_HOURS}) FROM raw_segments)
    AS excluded_over_cap
FROM segments
"""

cohort_comparison_sql = f"""
WITH
{BASE_SEGMENTS_CTE}
SELECT
  1 AS sort_order,
  'System-wide transfer cohort' AS cohort,
  COUNT(*) AS segment_n,
  COUNT(DISTINCT hadm_id) AS admission_n,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(50)] AS median_hours
FROM segments

UNION ALL

SELECT
  2 AS sort_order,
  'Admission-linked cohort' AS cohort,
  COUNT(*) AS segment_n,
  COUNT(DISTINCT hadm_id) AS admission_n,
  APPROX_QUANTILES(duration_hours, 100)[OFFSET(50)] AS median_hours
FROM admission_segments

ORDER BY sort_order
"""

threshold_sql = f"""
WITH
{BASE_SEGMENTS_CTE},
thresholds AS (
  SELECT * FROM UNNEST([
    STRUCT(48 AS threshold_hours, '>48 h' AS threshold_label, 1 AS sort_order),
    STRUCT(72 AS threshold_hours, '>72 h' AS threshold_label, 2 AS sort_order),
    STRUCT(168 AS threshold_hours, '>7 d' AS threshold_label, 3 AS sort_order),
    STRUCT(336 AS threshold_hours, '>14 d' AS threshold_label, 4 AS sort_order)
  ])
)
SELECT
  threshold_hours,
  threshold_label,
  sort_order,
  COUNTIF(duration_hours > threshold_hours) AS segment_n,
  100 * SAFE_DIVIDE(
    COUNTIF(duration_hours > threshold_hours),
    COUNT(*)
  ) AS pct_segments,
  100 * SAFE_DIVIDE(
    SUM(IF(duration_hours > threshold_hours, duration_hours, 0)),
    SUM(duration_hours)
  ) AS pct_total_hours
FROM segments
CROSS JOIN thresholds
GROUP BY threshold_hours, threshold_label, sort_order
ORDER BY sort_order
"""

careunit_sql = f"""
WITH
{BASE_SEGMENTS_CTE}
SELECT
  careunit,
  COUNT(*) AS n_segments_over_72h,
  SUM(GREATEST(duration_hours - {EXCESS_THRESHOLD_HOURS}, 0)) AS excess_72h
FROM segments
WHERE duration_hours > {EXCESS_THRESHOLD_HOURS}
GROUP BY careunit
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY excess_72h DESC
LIMIT {TOP_N_CAREUNITS}
"""

destination_sql = f"""
WITH
{BASE_SEGMENTS_CTE},
final_segments AS (
  SELECT *
  FROM admission_segments
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY hadm_id
    ORDER BY outtime DESC, intime DESC, transfer_id DESC
  ) = 1
),
service_at_discharge AS (
  SELECT
    a.hadm_id,
    (ARRAY_AGG(
        s.curr_service IGNORE NULLS
        ORDER BY s.transfertime DESC
        LIMIT 1
      )
    )[SAFE_OFFSET(0)] AS discharge_service
  FROM `{HOSP_DATASET}.admissions` a
  LEFT JOIN `{HOSP_DATASET}.services` s
    ON a.hadm_id = s.hadm_id
   AND s.transfertime <= a.dischtime
  GROUP BY a.hadm_id
),
labeled AS (
  SELECT
    f.hadm_id,
    f.duration_hours AS final_segment_hours,
    sd.discharge_service,
    CASE
      WHEN REGEXP_CONTAINS(
        UPPER(COALESCE(a.discharge_location, '')),
        r'HOSPICE'
      )
        THEN 'Hospice'
      WHEN UPPER(COALESCE(a.discharge_location, '')) = 'HOME'
        THEN 'Home'
      WHEN REGEXP_CONTAINS(
        UPPER(COALESCE(a.discharge_location, '')),
        r'HOME HEALTH'
      )
        THEN 'Home health'
      WHEN REGEXP_CONTAINS(
        UPPER(COALESCE(a.discharge_location, '')),
        r'SKILLED NURSING|REHAB|LONG TERM ACUTE|CHRONIC'
      )
        THEN 'Institutional post-acute'
      WHEN REGEXP_CONTAINS(
        UPPER(COALESCE(a.discharge_location, '')),
        r'ACUTE HOSPITAL'
      )
        THEN 'Acute hospital transfer'
      ELSE 'Other/unknown'
    END AS destination_category
  FROM final_segments f
  JOIN `{HOSP_DATASET}.admissions` a USING (hadm_id)
  LEFT JOIN service_at_discharge sd USING (hadm_id)
)
SELECT
  destination_category,
  COUNT(*) AS n,
  APPROX_QUANTILES(final_segment_hours, 100)[OFFSET(50)]
    AS median_final_hours,
  APPROX_QUANTILES(final_segment_hours, 100)[OFFSET(75)]
    AS p75_final_hours,
  100 * SAFE_DIVIDE(
    COUNTIF(final_segment_hours > 72),
    COUNT(*)
  ) AS pct_gt_72h
FROM labeled
WHERE discharge_service = 'MED'
  AND destination_category IN (
    'Home',
    'Home health',
    'Institutional post-acute',
    'Hospice',
    'Acute hospital transfer'
  )
GROUP BY destination_category
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY median_final_hours DESC
"""

kpi = query_df(kpi_sql)
cohort_comparison = query_df(cohort_comparison_sql)
thresholds = query_df(threshold_sql)
careunits = query_df(careunit_sql)
destinations = query_df(destination_sql)

assert len(kpi) == 1
assert set(cohort_comparison["cohort"]) == {
    "System-wide transfer cohort",
    "Admission-linked cohort",
}
assert careunits["n_segments_over_72h"].ge(MIN_CELL_N).all()
assert destinations["n"].ge(MIN_CELL_N).all()

clear_output(wait=True)

display(Markdown(
    """# Hospital Flow Burden in MIMIC-IV

## What this project does

This project uses hospital transfer and discharge records from **MIMIC-IV v3.1**
to examine how long patients remain in individual care units at Beth Israel
Deaconess Medical Center. The goal is to identify where unusually long
care-unit segments consume capacity and where an operations team should
investigate first.

It answers three practical questions:

1. **How much of the hospital's recorded care-unit time is concentrated in long segments?**
2. **Which care units contribute the most time beyond a 72-hour threshold?**
3. **Do discharge destinations provide a signal that downstream-care access may be associated with longer final segments?**

The analysis identifies the **location and scale of operational burden**.
It does not determine when a patient was medically ready for discharge or which
hours were avoidable.
"""
))

row = kpi.iloc[0]
t72 = thresholds.loc[thresholds["threshold_hours"] == 72].iloc[0]
top_unit = (
    careunits.iloc[0]["careunit"]
    if not careunits.empty
    else "No publishable care unit"
)

display(Markdown(
    "## Executive finding\n\n"
    f"**The burden is concentrated in the long tail.** "
    f"{float(t72['pct_segments']):.1f}% of valid system-wide care-unit "
    f"segments exceed 72 hours, but those segments account for "
    f"{float(t72['pct_total_hours']):.1f}% of all recorded segment-hours. "
    f"The largest contributor to excess time beyond 72 hours is "
    f"**{html.escape(str(top_unit))}**. "
    f"Among admissions whose last recorded service was MED, the median final segment was 57.6 hours longer for institutional post-acute destinations than for home discharge."
))

cards = [
    ("System-wide median segment", f"{float(row['median_hours']):.1f} h"),
    ("Segments >48 h", f"{float(row['pct_gt_48h']):.1f}%"),
    ("Segments >72 h", f"{float(row['pct_gt_72h']):.1f}%"),
    ("Segments >7 d", f"{float(row['pct_gt_7d']):.1f}%"),
]

card_html = """
<div style="display:flex;gap:12px;flex-wrap:wrap;margin:14px 0 8px 0;">
""" + "".join(
    f"""
    <div style="min-width:155px;flex:1;border:1px solid #d9d9d9;border-radius:8px;padding:12px 14px;">
      <div style="font-size:12px;opacity:.70;">{html.escape(label)}</div>
      <div style="font-size:25px;font-weight:700;line-height:1.2;">{html.escape(value)}</div>
    </div>
    """
    for label, value in cards
) + "</div>"

display(HTML(card_html))

display(Markdown(
    f"<small>System-wide scope: {format_n(row['segment_n'])} valid segments. "
    f"{format_n(row['admission_linked_segment_n'])} are linked to "
    f"{format_n(row['admission_n'])} hospital admissions; "
    f"{format_n(row['non_admission_segment_n'])} have no `hadm_id`. "
    f"{format_n(row['excluded_over_cap'])} segment(s) over 365 days were "
    f"excluded as probable administrative artifacts.</small>"
))

cohort_indexed = cohort_comparison.set_index("cohort")
system_median = float(
    cohort_indexed.loc["System-wide transfer cohort", "median_hours"]
)
linked_median = float(
    cohort_indexed.loc["Admission-linked cohort", "median_hours"]
)

display(Markdown(
    "The median describes the typical system-wide care-unit segment, while "
    "the threshold KPIs show how much activity falls into the long right "
    "tail. The difference between the median and the long-stay percentages "
    "is the central operational story: most segments are relatively short, "
    "but a meaningful minority remain for days."
))

display(Markdown(
    """## 1. Long segments consume a disproportionate share of capacity

The first graph compares two quantities at each duration threshold:

- **Share of segments:** the percentage of care-unit segments lasting longer than the threshold.
- **Share of segment-hours:** the percentage of all recorded care-unit time consumed by those segments.

When the segment-hours bar is much taller than the segment-share bar, a small
group of long segments is consuming a disproportionate amount of capacity.
"""
))

plot_df = thresholds.sort_values("sort_order").copy()
x = np.arange(len(plot_df))
width = 0.36

fig, ax = plt.subplots()
bars1 = ax.bar(
    x - width / 2,
    plot_df["pct_segments"],
    width,
    label="Share of segments",
)
bars2 = ax.bar(
    x + width / 2,
    plot_df["pct_total_hours"],
    width,
    label="Share of segment-hours",
)

ax.set_title(
    "A minority of segments consumes a disproportionate share of time"
)
ax.set_ylabel("Percent")
ax.set_xticks(x, plot_df["threshold_label"])
ax.set_ylim(0, max(float(plot_df["pct_total_hours"].max()) * 1.18, 10))
ax.grid(axis="y", alpha=0.25)
ax.legend(frameon=False)

ax.bar_label(
    bars1,
    labels=[f"{v:.1f}%" for v in plot_df["pct_segments"]],
    padding=3,
    fontsize=9,
)
ax.bar_label(
    bars2,
    labels=[f"{v:.1f}%" for v in plot_df["pct_total_hours"]],
    padding=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

display(Markdown(
    f"At the 72-hour threshold, **{float(t72['pct_segments']):.1f}%** of "
    f"segments consume **{float(t72['pct_total_hours']):.1f}%** of all "
    "recorded segment-hours. This indicates that the main capacity burden "
    "is concentrated in the right tail rather than distributed evenly "
    "across all segments."
))

display(Markdown(
    """## 2. The burden is concentrated in specific care units

The second graph ranks care units by total hours accumulated after the first
72 hours of each long segment. The first 72 hours are not counted as excess.

A high rank identifies where the long-duration burden is located. It does not
by itself establish inefficiency or avoidable delay; case mix, clinical
complexity, transfer practices, transport, payer processes, and downstream
capacity may all contribute.
"""
))

unit_plot = careunits.copy()
unit_plot["label"] = unit_plot["careunit"].map(
    lambda x: textwrap.fill(str(x), width=26)
)
unit_plot = unit_plot.sort_values("excess_72h")

fig, ax = plt.subplots(figsize=(9, 5.4))
bars = ax.barh(
    unit_plot["label"],
    unit_plot["excess_72h"],
)
ax.set_title(
    "Excess time beyond 72 hours is concentrated in a few care units"
)
ax.set_xlabel("Excess care-unit hours beyond 72 h (millions)")
ax.grid(axis="x", alpha=0.25)
ax.bar_label(
    bars,
    labels=[
        f"{v/1000000:,.1f}M"
        for v in unit_plot["excess_72h"]
    ],
    padding=3,
    fontsize=9,
)

plt.tight_layout()
plt.show()

display(Markdown(
    f"**{html.escape(str(top_unit))}** contributes the most accumulated "
    "time beyond 72 hours in the system-wide transfer cohort and is the "
    "first place an operations team should investigate."
))

display(Markdown(
    """## 3. Discharge destination provides a downstream-care signal

This analysis now switches to the **admission-linked cohort**, because a
hospital admission ID is required to identify the final recorded care-unit
segment, the service responsible at discharge, and the discharge destination.

The comparison is limited to admissions whose last recorded service is
general medicine (`MED`). It uses the direct median of final segments within
each destination category.
"""
))

if destinations.empty:
    display(Markdown(
        "**Downstream-care signal:** no destination category met the "
        "publication threshold within general medicine."
    ))
else:
    indexed = destinations.set_index("destination_category")

    if {"Home", "Institutional post-acute"}.issubset(indexed.index):
        home_med = float(indexed.loc["Home", "median_final_hours"])
        post_med = float(
            indexed.loc[
                "Institutional post-acute",
                "median_final_hours",
            ]
        )
        delta = post_med - home_med

        if delta > 0:
            destination_message = (
                "**Supporting signal:** within admissions discharged from "
                "general medicine, the median final segment is "
                f"**{delta:.1f} hours longer** for institutional post-acute "
                "destinations than for home discharge."
            )
        else:
            destination_message = (
                "**Supporting signal:** within general medicine, "
                "institutional post-acute discharges do not have a longer "
                "median final segment than home discharges in this extract."
            )
    else:
        destination_message = (
            "**Supporting signal:** final-segment duration varies by "
            "discharge destination within general medicine."
        )

    display(Markdown(destination_message))

    dest_plot = destinations.sort_values("median_final_hours").copy()
    dest_plot["label"] = dest_plot["destination_category"].map(
        lambda x: textwrap.fill(str(x), width=24)
    )

    fig, ax = plt.subplots(figsize=(9, 4.8))
bars = ax.barh(
        dest_plot["label"],
        dest_plot["median_final_hours"],
    )
ax.set_title(
        "Final segment before discharge, general medicine only"
    )
ax.set_xlabel("Median final-segment duration (hours)")
ax.grid(axis="x", alpha=0.25)

labels = [
        f"{hours:.1f} h | n={format_n(n)}"
        for hours, n in zip(
            dest_plot["median_final_hours"],
            dest_plot["n"],
        )
    ]
ax.bar_label(bars, labels=labels, padding=3, fontsize=9)

plt.tight_layout()
plt.show()

postacute_sentence = ""
if not destinations.empty:
    indexed = destinations.set_index("destination_category")
    if {"Home", "Institutional post-acute"}.issubset(indexed.index):
        if (
            indexed.loc[
                "Institutional post-acute",
                "median_final_hours",
            ]
            > indexed.loc["Home", "median_final_hours"]
        ):
            postacute_sentence = (
                "The longer final segment before institutional post-acute "
                "discharge is consistent with downstream capacity friction, "
                "but it is not proof of avoidable delay."
            )
        else:
            postacute_sentence = (
                "This extract does not show a longer median final segment "
                "for institutional post-acute discharge than for home "
                "discharge."
            )

display(Markdown(
    f"""## Bottom line

The operational problem is not the typical segment; it is the
**small, resource-intensive right tail**.

The first place to investigate is **{html.escape(str(top_unit))}**, because
it contributes the most excess time beyond 72 hours in the complete
system-wide transfer cohort.

Among admissions whose last recorded service was MED, the median final segment was 57.6 hours longer for institutional post-acute destinations than for home discharge.

{postacute_sentence}

**Next data needed:** discharge-ready time, referral time, payer
authorization, placement acceptance, downstream bed availability, transport
booking, and actual departure. MIMIC can locate burden, but it cannot
determine which hours were avoidable."""
))

<details>
<summary><strong>Method, cohort, privacy, and interpretation notes</strong></summary>

- **Headline/system-wide cohort:** every positive-duration row in `hosp.transfers` with non-null `careunit`, `intime`, and `outtime`, and duration no longer than 365 days. `hadm_id` may be null.
- **Admission-linked cohort:** the same cleaned transfer rows restricted to `hadm_id IS NOT NULL`. This cohort is used only where hospitalization linkage is required.
- The headline median is **8.8 hours** because the main analysis uses the complete system-wide transfer cohort, including ED-only segments without a hospital admission ID. Restricting the same cleaning rules to `hadm_id IS NOT NULL` produces an admission-linked  median of 16.6 hours. That narrower cohort is used  only when hospitalization context is required, such as the final-unit and discharge-destination analysis.
- ED-only transfer rows without a hospital admission ID remain in the main hospital-wide distribution. Excluding them changes the median and creates the earlier 8.8-hour versus 16.6-hour discrepancy.
- Durations above 365 days are excluded as probable administrative artifacts.
- Quantiles are calculated with BigQuery `APPROX_QUANTILES`.
- “Excess time beyond 72 hours” is `max(duration − 72, 0)` summed across segments.
- The discharge-destination comparison uses the **direct median of final segments** among admissions whose last recorded service is `MED`; it does not average subgroup medians.
- Home health, hospice, acute-hospital transfer, and institutional post-acute care remain separate.
- Every grouped output requires `n ≥ 11`; no patient-level or row-level derived datasets are produced or shared. Only aggregate outputs with n ≥ 11 are published.
- This is a single-center observational EHR analysis. Segment duration is not the same as confirmed discharge delay, medical readiness, avoidability, or causal impact.
- Source: MIMIC-IV v3.1, Beth Israel Deaconess Medical Center, 2008–2022.  
  Johnson et al. (2024), *MIMIC-IV v3.1*, PhysioNet, DOI: `10.13026/kpb9-mt58`.  
  Johnson et al. (2023), *Scientific Data*, DOI: `10.1038/s41597-022-01899-x`.  
  Goldberger et al. (2000), *Circulation*, 101(23), e215–e220.

</details>